In [3]:
# ============================================================
# Phase-5 EXPORTER (UPDATED) — adds Commit supervision metadata
# ------------------------------------------------------------
# What this update does:
#   ✅ Exports these fields into each shard NPZ so your loader can build y_ready/commit_mask:
#      - dist_to_onset_s, onset_idx_nearest, onset_time_nearest_s
#      - active_prob_mean, w_label_mean
#      - is_transition  (prefer existing; else derived from trans_frac)
#      - win_idx        (per (subject_id, task_code, trial_id) chronological index)
#
# IMPORTANT:
#   - You do NOT need to update Phase-3 for this (Phase-3 already writes these cols).
#   - You MUST re-run Phase-5 after this to regenerate exports.
#   - I recommend bumping P5_VERSION so you don't overwrite old exports.
# ============================================================

from __future__ import annotations
import json
import hashlib
import warnings
import re
from pathlib import Path
from collections import OrderedDict, defaultdict
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
ROOT_DIR    = Path(r"/home/tsultan1/IROS/Data")
DATASET_DIR = ROOT_DIR / "_dataset_icml_v1"

# Phase-3 splits
SPLITS_CSV  = DATASET_DIR / "splits_v3.csv"   # change if needed

# ---- Phase 4 cache layout (recommended hashed dir) ----
CACHE_DIR_HASHED = DATASET_DIR / "preproc_v4_2"    # optional hashed cache folder
USE_HASHED_CACHE_FIRST = True

# ---- Phase 4.5 QC ----
QC_POLICY = "core_ok"   # "core_ok" | "tri_good" | "off"

QC_SUMMARY_V4 = DATASET_DIR / "qc_summary_v4.csv"
QC_ALLOW_CORE_OK  = DATASET_DIR / "qc_allowlist_v4_passwarn.csv"
QC_ALLOW_TRI_GOOD = DATASET_DIR / "qc_allowlist_v4_trimodal_good.csv"  # if you have it

# ---- Outputs (bump version) ----
EXPORT_BALANCED   = True
EXPORT_SSL        = True
EXPORT_LIVE       = False   # set True to export causal windows ("live")

# ✅ bump so you don't overwrite old exports
P5_VERSION        = "v6_3c_commitmeta"
OUT_NAME_BALANCED = f"exports_{P5_VERSION}_balanced_{QC_POLICY}"
OUT_NAME_SSL      = f"exports_{P5_VERSION}_ssl_{QC_POLICY}"

# For live export variants (suffix appended)
LIVE_SUFFIX = "_live"

# ---- TRAIN balancing (balanced export only) ----
TARGET_TASK_PER_CLASS = 1200
TASK_PER_CLASS_CAP    = 4000

TARGET_REST_FRACTION  = 0.40
MIN_REST_SAMPLES      = 1500

MAX_ANCHOR_RATIO      = 0.35  # anchors <= ratio * (#task_valid)

# ---- Window Types ----
WINDOW_TYPES_MAIN = ["sliding_action_safe", "sliding_rest_safe"]
INCLUDE_ONSET_ANCHOR = True
WINDOW_TYPES_EXTRA = ["onset_anchor"] if INCLUDE_ONSET_ANCHOR else []
WINDOW_TYPES_EXPORT = WINDOW_TYPES_MAIN + WINDOW_TYPES_EXTRA

# ---- Modalities ----
INCLUDE_EEG, INCLUDE_EMG, INCLUDE_ET = True, True, True

# ---- Shards / windows ----
SHARD_SIZE        = 5000
FIXED_WINDOW_LEN  = 500   # 2.0s at 250 Hz

# Coverage check computed over PRESENT channels only
MIN_COVERAGE      = 0.40
MIN_MODALITIES_OK = 1

# ---- Quality-aware knobs ----
Q_MIN_BALANCE = 0.55
Q_ALPHA       = 2.0

# ---- Debug limits ----
LIMIT_TRAIN = None
LIMIT_VAL   = None
LIMIT_TEST  = None

# ---------------- NPZ (Phase-4 caches) ----------------
NPZ_SUFFIX_CANDIDATES = [
    ".preproc.v4_3.npz",
    ".preproc.v4.3.npz",
    ".preproc.v4_2.npz",
    ".preproc.v4.2.npz",
    ".preproc.v4_1.npz",
    ".preproc.v4.1.npz",
    ".preproc.v4.npz",
    ".preproc.v3.npz",
    ".preproc.v2.npz",
    ".preproc.npz",
]
NPZ_SUFFIX_PREFERRED = ".preproc.v4_2.npz"

EEG_KEY      = "EEG"
EEG_MASK_KEY = "EEG_mask"
EEG_CH_KEY   = "EEG_ch"

EMG_KEY      = "EMG_env"
EMG_MASK_KEY = "EMG_mask"
EMG_CH_KEY   = "EMG_ch"

ET_KEY       = "ET"
ET_CH_KEY    = "ET_ch"
ET_VALID_KEY = "ET_valid"
ET_MASK_KEYS = ["ET_mask", "ET_feat_mask"]

# ============================================================
# ✅ NEW: Commit-meta export controls
# ============================================================
EXPORT_COMMIT_META = True
TRANS_FRAC_TO_TRANSITION_THR = 1e-6  # if trans_frac > thr -> is_transition=1
WINIDX_GROUP_COLS = ["subject_id", "task_code", "trial_id"]
WINIDX_TIME_COL_CANDIDATES = ["center_time_s", "start_time_s", "start_idx", "end_idx"]


# =============================================================================
# Utilities
# =============================================================================
def p5_mkdir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def p5_md5_str(s: str) -> str:
    return hashlib.md5(s.encode("utf-8")).hexdigest()

def p5_norm_base(path_like: str) -> str:
    s = str(path_like)
    s = re.sub(r"\.preproc\..*\.npz$", "", s)
    if s.endswith(".npz"):
        for suf in NPZ_SUFFIX_CANDIDATES:
            if s.endswith(suf):
                s = s[:-len(suf)]
                break
    if s.endswith(".csv"):
        s = s[:-4]
    return s

def p5_read_npz(npz_path: Path) -> dict:
    with np.load(npz_path, allow_pickle=True) as z:
        out = {k: z[k] for k in z.files}
    for ch_key in (EEG_CH_KEY, EMG_CH_KEY, ET_CH_KEY):
        if ch_key in out:
            out[ch_key] = [str(x) for x in out[ch_key].tolist()]
    return out

def p5_limit_for_split(split: str):
    if split == "train" and LIMIT_TRAIN:
        return LIMIT_TRAIN
    if split == "val" and LIMIT_VAL:
        return LIMIT_VAL
    if split == "test" and LIMIT_TEST:
        return LIMIT_TEST
    return None

def p5_slice_time(arr: np.ndarray, s: int, e: int) -> np.ndarray:
    s = max(0, int(s))
    e = min(int(e), arr.shape[0])
    if e <= s:
        return arr[:0]
    return arr[s:e]

def p5_init_stats(nc: int) -> dict:
    return {"cnt": np.zeros(nc, np.float64), "sum": np.zeros(nc, np.float64), "sum2": np.zeros(nc, np.float64)}

def p5_add_stats(stats: dict, X: np.ndarray, M: np.ndarray) -> None:
    if X is None or M is None or X.size == 0 or M.size == 0:
        return
    X = X.astype(np.float32, copy=False)
    m = M.astype(np.float32, copy=False)
    finite = np.isfinite(X).astype(np.float32)
    m = m * finite
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    cnt = m.sum(axis=0)
    if float(cnt.sum()) <= 0.0:
        return
    stats["cnt"] += cnt
    stats["sum"] += (X * m).sum(axis=0)
    stats["sum2"] += ((X ** 2) * m).sum(axis=0)

def p5_finalize_stats(stats: dict) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    cnt = stats["cnt"].astype(np.float64)
    mean = np.zeros_like(cnt, dtype=np.float64)
    std  = np.ones_like(cnt, dtype=np.float64)
    seen = cnt > 0.0
    if np.any(seen):
        mean[seen] = stats["sum"][seen] / cnt[seen]
        var = stats["sum2"][seen] / cnt[seen] - mean[seen] ** 2
        var = np.maximum(var, 1e-12)
        std[seen] = np.sqrt(var)
    std = std + 1e-8
    return mean.astype(np.float32), std.astype(np.float32), cnt.astype(np.float32)

def p5_zscore_masked(X: np.ndarray, M: np.ndarray, mean: np.ndarray, std: np.ndarray) -> np.ndarray:
    if X is None or M is None or X.size == 0:
        return np.zeros((0, 0), dtype=np.float32)
    Y = (X - mean[None, :]) / std[None, :]
    Y[M <= 0.0] = 0.0
    return np.nan_to_num(Y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def p5_pad_or_trim(X: np.ndarray, target_len: int) -> np.ndarray:
    if X is None or X.size == 0:
        return np.zeros((target_len, 0), dtype=np.float32)
    T, C = X.shape
    if T == target_len:
        return X.astype(np.float32, copy=False)
    if T > target_len:
        return X[:target_len].astype(np.float32, copy=False)
    pad = np.zeros((target_len - T, C), dtype=np.float32)
    return np.concatenate([X.astype(np.float32, copy=False), pad], axis=0)

def p5_pad_or_trim_mask(M: np.ndarray, target_len: int) -> np.ndarray:
    if M is None or M.size == 0:
        return np.zeros((target_len, 0), dtype=np.float32)
    T, C = M.shape
    if T == target_len:
        return M.astype(np.float32, copy=False)
    if T > target_len:
        return M[:target_len].astype(np.float32, copy=False)
    pad = np.zeros((target_len - T, C), dtype=np.float32)
    return np.concatenate([M.astype(np.float32, copy=False), pad], axis=0)

def p5_qscore_from_modalities(r_eeg: float, r_emg: float, r_et: float, kept_mask: int, qc_scale: float) -> float:
    rs = []
    if kept_mask & 1: rs.append(float(r_eeg))
    if kept_mask & 2: rs.append(float(r_emg))
    if kept_mask & 4: rs.append(float(r_et))
    if not rs:
        return 0.0
    return float(np.mean(rs) * float(qc_scale))

# ============================================================
# ✅ NEW: Commit-meta helpers
# ============================================================
def _ensure_col(df: pd.DataFrame, col: str, default):
    if col not in df.columns:
        df[col] = default

def p5_prepare_commit_meta_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensures commit-related meta exists and adds:
      - win_idx (per trial)
      - is_transition (prefer col; else derived from trans_frac)
    """
    if not EXPORT_COMMIT_META:
        return df

    df = df.copy()

    # ensure optional cols exist (Phase-3 writes most of these already)
    _ensure_col(df, "dist_to_onset_s", np.nan)
    _ensure_col(df, "onset_idx_nearest", -1)
    _ensure_col(df, "onset_time_nearest_s", np.nan)
    _ensure_col(df, "active_prob_mean", np.nan)
    _ensure_col(df, "w_label_mean", np.nan)
    _ensure_col(df, "trans_frac", np.nan)

    # derive is_transition if missing
    if "is_transition" not in df.columns:
        tf = pd.to_numeric(df["trans_frac"], errors="coerce").fillna(0.0).to_numpy(np.float32, copy=False)
        df["is_transition"] = (tf > float(TRANS_FRAC_TO_TRANSITION_THR)).astype(np.int8)
    else:
        it = pd.to_numeric(df["is_transition"], errors="coerce").fillna(0).astype(np.int8)
        df["is_transition"] = it

    # choose a time/order column for win_idx
    time_col = None
    for c in WINIDX_TIME_COL_CANDIDATES:
        if c in df.columns:
            time_col = c
            break
    if time_col is None:
        # ultra-safe
        df["_ord"] = np.arange(len(df), dtype=np.int64)
        time_col = "_ord"

    # stable sort (mergesort preserves earlier ordering deterministically)
    sort_cols = [c for c in WINIDX_GROUP_COLS if c in df.columns] + [time_col, "type"]
    sort_cols = [c for c in sort_cols if c in df.columns]
    df = df.sort_values(sort_cols, kind="mergesort").reset_index(drop=True)

    # win_idx within trial
    if all(c in df.columns for c in WINIDX_GROUP_COLS):
        df["win_idx"] = df.groupby(WINIDX_GROUP_COLS, sort=False).cumcount().astype(np.int32)
    else:
        df["win_idx"] = np.arange(len(df), dtype=np.int32)

    # clean numeric types
    df["dist_to_onset_s"] = pd.to_numeric(df["dist_to_onset_s"], errors="coerce").astype(np.float32)
    df["onset_idx_nearest"] = pd.to_numeric(df["onset_idx_nearest"], errors="coerce").fillna(-1).astype(np.int32)
    df["onset_time_nearest_s"] = pd.to_numeric(df["onset_time_nearest_s"], errors="coerce").astype(np.float32)
    df["active_prob_mean"] = pd.to_numeric(df["active_prob_mean"], errors="coerce").astype(np.float32)
    df["w_label_mean"] = pd.to_numeric(df["w_label_mean"], errors="coerce").astype(np.float32)

    return df


# =============================================================================
# Cache LRU
# =============================================================================
class P5CacheLRU:
    def __init__(self, capacity: int = 16):
        self.capacity = int(capacity)
        self._d: OrderedDict[str, dict] = OrderedDict()

    def _resolve_npz_path(self, label_csv: str, cache_npz: Optional[str] = None) -> Path:
        label_csv = str(label_csv)

        if cache_npz is not None:
            p = Path(str(cache_npz))
            if p.exists():
                return p

        if USE_HASHED_CACHE_FIRST and CACHE_DIR_HASHED.exists():
            h = p5_md5_str(label_csv)
            cand = CACHE_DIR_HASHED / f"{h}{NPZ_SUFFIX_PREFERRED}"
            if cand.exists():
                return cand
            for suf in NPZ_SUFFIX_CANDIDATES:
                cand2 = CACHE_DIR_HASHED / f"{h}{suf}"
                if cand2.exists():
                    return cand2

        cand = Path(label_csv + NPZ_SUFFIX_PREFERRED)
        if cand.exists():
            return cand
        for suf in NPZ_SUFFIX_CANDIDATES:
            cand2 = Path(label_csv + suf)
            if cand2.exists():
                return cand2

        raise FileNotFoundError(
            f"cache missing for:\n  {label_csv}\n"
            f"tried: explicit cache_npz (if any), hashed: {CACHE_DIR_HASHED}/<md5><suffix>, "
            f"sidecar: {label_csv}<suffix>\n"
            f"suffixes tried: {NPZ_SUFFIX_CANDIDATES}"
        )

    def fetch(self, label_csv: str, cache_npz: Optional[str] = None) -> dict:
        key = f"{label_csv}||{cache_npz or ''}"
        if key in self._d:
            self._d.move_to_end(key)
            return self._d[key]

        npz_path = self._resolve_npz_path(label_csv, cache_npz=cache_npz)
        val = p5_read_npz(npz_path)

        self._d[key] = val
        self._d.move_to_end(key)
        if len(self._d) > self.capacity:
            self._d.popitem(last=False)
        return val

    def reset(self) -> None:
        self._d.clear()


# =============================================================================
# Schema + alignment
# =============================================================================
def p5_sort_channels(modality: str, ch_list: List[str]) -> List[str]:
    def _num_tail(name: str, prefix: str):
        if name.startswith(prefix):
            tail = name[len(prefix):]
            if tail.isdigit():
                return int(tail)
        return None

    if modality.upper() == "EEG":
        eeg_num, other = [], []
        for c in ch_list:
            n = _num_tail(c, "EEG_Ch")
            (eeg_num if n is not None else other).append((n, c) if n is not None else c)
        eeg_num_sorted = [c for _, c in sorted(eeg_num, key=lambda x: x[0])]
        return eeg_num_sorted + sorted(other)

    if modality.upper() == "EMG":
        emg_num, other = [], []
        for c in ch_list:
            n = _num_tail(c, "EMG_Ch")
            (emg_num if n is not None else other).append((n, c) if n is not None else c)
        emg_num_sorted = [c for _, c in sorted(emg_num, key=lambda x: x[0])]
        return emg_num_sorted + sorted(other)

    groups = [
        ("ET_Gaze", 0), ("ET_Pupil", 1), ("ET_Distance", 2),
        ("ET_Gyro", 3), ("ET_Acc", 4), ("ET_HeadRotation", 5), ("ET_", 9),
    ]
    def _et_key(c: str):
        g = 99
        for prefix, idx in groups:
            if c.startswith(prefix):
                g = idx
                break
        return (g, c)

    return [c for c in sorted(ch_list, key=_et_key)]

def p5_get_mask_from_npz(npz: dict, keys: List[str]) -> Optional[np.ndarray]:
    for k in keys:
        if k in npz:
            return npz[k]
    return None

def p5_align_window_to_schema(
    npz: dict,
    data_key: str,
    mask_keys: List[str],
    ch_key: str,
    schema_ch: List[str],
    s: int,
    e: int,
) -> Tuple[Optional[np.ndarray], Optional[np.ndarray], np.ndarray]:
    C = len(schema_ch)
    present = np.zeros(C, dtype=np.float32)

    if data_key not in npz or ch_key not in npz:
        return None, None, present

    Mfull = p5_get_mask_from_npz(npz, mask_keys)
    if Mfull is None:
        return None, None, present

    X = npz[data_key]
    M = Mfull
    file_ch = npz[ch_key]

    Xw = p5_slice_time(X, s, e)
    Mw = p5_slice_time(M, s, e)
    if Xw.size == 0 or Mw.size == 0:
        return None, None, present

    idx_map = {c: i for i, c in enumerate(file_ch)}
    T = Xw.shape[0]
    X_al = np.zeros((T, C), dtype=np.float32)
    M_al = np.zeros((T, C), dtype=np.float32)

    for j, cname in enumerate(schema_ch):
        if cname in idx_map:
            i = idx_map[cname]
            if i < Xw.shape[1] and i < Mw.shape[1]:
                X_al[:, j] = Xw[:, i].astype(np.float32, copy=False)
                M_al[:, j] = Mw[:, i].astype(np.float32, copy=False)
                present[j] = 1.0

    return X_al, M_al, present

def p5_quality_from_mask_present_only(M: np.ndarray, present_mask: np.ndarray) -> float:
    if M is None or M.size == 0:
        return 0.0
    idx = np.where(present_mask > 0.5)[0]
    if idx.size == 0:
        return 0.0
    return float(M[:, idx].mean())

def p5_present_fraction(present_mask: np.ndarray) -> float:
    if present_mask.size == 0:
        return 0.0
    return float(present_mask.mean())

def p5_coverage_ok_present_only(M: np.ndarray, present_mask: np.ndarray, thr: float) -> bool:
    if M is None or M.size == 0:
        return True
    idx = np.where(present_mask > 0.5)[0]
    if idx.size == 0:
        return False
    cov = M[:, idx].mean(axis=0)
    return bool((cov >= thr).mean() > 0.5)

def p5_build_fold_schema(fold_rows: pd.DataFrame, cache: P5CacheLRU) -> dict:
    schema = {"EEG": [], "EMG": [], "ET": []}
    seen = {"EEG": set(), "EMG": set(), "ET": set()}

    unique_pairs = (
        fold_rows[["file"] + (["cache_npz"] if "cache_npz" in fold_rows.columns else [])]
        .drop_duplicates()
        .values
        .tolist()
    )

    for pair in unique_pairs:
        label_csv = str(pair[0])
        cache_npz = str(pair[1]) if (len(pair) > 1 and pd.notna(pair[1])) else None
        npz = cache.fetch(label_csv, cache_npz=cache_npz)

        if INCLUDE_EEG and EEG_CH_KEY in npz:
            for c in npz[EEG_CH_KEY]:
                if c not in seen["EEG"]:
                    seen["EEG"].add(c); schema["EEG"].append(c)

        if INCLUDE_EMG and EMG_CH_KEY in npz:
            for c in npz[EMG_CH_KEY]:
                if c not in seen["EMG"]:
                    seen["EMG"].add(c); schema["EMG"].append(c)

        if INCLUDE_ET and ET_CH_KEY in npz:
            for c in npz[ET_CH_KEY]:
                if c not in seen["ET"]:
                    seen["ET"].add(c); schema["ET"].append(c)

    schema["EEG"] = p5_sort_channels("EEG", schema["EEG"])
    schema["EMG"] = p5_sort_channels("EMG", schema["EMG"])
    schema["ET"]  = p5_sort_channels("ET",  schema["ET"])
    return schema


# =============================================================================
# Window bounds
# =============================================================================
def p5_get_window_bounds(npz: dict, row: Any, causal: bool = False) -> Tuple[int, int]:
    t = npz.get("t", None)
    has_t = isinstance(t, np.ndarray) and t.ndim == 1 and t.size > 0

    for (a, b) in [("start_time_s", "end_time_s"), ("start_time", "end_time")]:
        if hasattr(row, a) and hasattr(row, b) and has_t:
            ts = float(getattr(row, a))
            te = float(getattr(row, b))
            if np.isfinite(ts) and np.isfinite(te) and te > ts:
                s = int(np.searchsorted(t, ts, side="left"))
                e = int(np.searchsorted(t, te, side="right"))
                if causal:
                    e = max(0, min(e, int(t.size)))
                    s = max(0, e - int(FIXED_WINDOW_LEN))
                return s, e

    s = int(getattr(row, "start_idx"))
    e = int(getattr(row, "end_idx"))
    if causal:
        e = max(0, e)
        s = max(0, e - int(FIXED_WINDOW_LEN))
    return s, e


# =============================================================================
# QC integration (unchanged)
# =============================================================================
def load_qc_maps() -> Tuple[Optional[set], Dict[str, int], Dict[str, str]]:
    pol = QC_POLICY.lower()
    if pol == "off":
        return None, {}, {}

    if pol == "core_ok":
        allow_path = QC_ALLOW_CORE_OK
    elif pol == "tri_good":
        allow_path = QC_ALLOW_TRI_GOOD
    else:
        raise ValueError(f"QC_POLICY must be one of: core_ok | tri_good | off  (got: {QC_POLICY})")

    allowset_base = None
    qc_flag_map: Dict[str, int] = {}
    qc_cache_map: Dict[str, str] = {}

    if allow_path.exists():
        al = pd.read_csv(allow_path)
        col = "label_csv" if "label_csv" in al.columns else ("file" if "file" in al.columns else al.columns[0])
        allowset_base = set(al[col].astype(str).apply(p5_norm_base).tolist())
        print(f"[QC] Using allowlist ({QC_POLICY}): {allow_path}  (N={len(allowset_base)})")
    else:
        print(f"[QC] allowlist not found: {allow_path} → will fallback to qc_summary rule if possible.")
        allowset_base = None

    if QC_SUMMARY_V4.exists():
        qs = pd.read_csv(QC_SUMMARY_V4)
        col = "label_csv" if "label_csv" in qs.columns else ("file" if "file" in qs.columns else qs.columns[0])
        base_col = qs[col].astype(str).apply(p5_norm_base)

        if "qc_status" in qs.columns:
            st = qs["qc_status"].astype(str).str.upper()
            for b, s in zip(base_col.tolist(), st.tolist()):
                if s == "PASS":
                    qc_flag_map[b] = 2
                elif s == "WARN":
                    qc_flag_map[b] = 1
                else:
                    qc_flag_map[b] = 0
            print(f"[QC] qc_flag mapping loaded from qc_summary_v4.csv (N={len(qc_flag_map)})")
        else:
            print("[QC] qc_summary_v4.csv missing qc_status → qc_flag map empty.")

        if "cache_npz" in qs.columns:
            for b, p in zip(base_col.tolist(), qs["cache_npz"].astype(str).tolist()):
                if p and p != "nan":
                    qc_cache_map[b] = p
            print(f"[QC] qc_cache_npz map loaded from qc_summary_v4.csv (N={len(qc_cache_map)})")

        if allowset_base is None and "qc_status" in qs.columns:
            st = qs["qc_status"].astype(str).str.upper()
            if pol == "core_ok":
                keep = st.isin(["PASS", "WARN"])
                allowset_base = set(base_col[keep].tolist())
                print(f"[QC] Fallback core_ok allowset from qc_summary: N={len(allowset_base)}")
            else:
                keep = st.isin(["PASS", "WARN"])
                et_cols = [c for c in qs.columns if ("et" in c.lower() and "strict" in c.lower() and "cov" in c.lower())]
                if not et_cols:
                    et_cols = [c for c in qs.columns if ("et" in c.lower() and "strict" in c.lower() and "mean" in c.lower())]
                if et_cols:
                    c0 = et_cols[0]
                    etv = pd.to_numeric(qs[c0], errors="coerce").fillna(0.0)
                    keep = keep & (etv >= 0.30)
                    allowset_base = set(base_col[keep].tolist())
                    print(f"[QC] Fallback tri_good allowset using {c0}>=0.30: N={len(allowset_base)}")
                else:
                    allowset_base = set(base_col[keep].tolist())
                    print(f"[QC] tri_good allowlist missing and no ET strict column found → fallback to PASS/WARN: N={len(allowset_base)}")
    else:
        print("[QC] qc_summary_v4.csv not found → qc_flag map empty; no fallback allowset.")
        if allowset_base is None:
            allowset_base = None

    return allowset_base, qc_flag_map, qc_cache_map


# =============================================================================
# Task-valid logic
# =============================================================================
def infer_use_for_task(df: pd.DataFrame) -> pd.Series:
    if "use_for_task" in df.columns:
        return (pd.to_numeric(df["use_for_task"], errors="coerce").fillna(0).astype(int) > 0)
    return (
        (df["label_action"].astype(int) == 1)
        & (df["task_target"].astype(int) > 0)
        & (df["type"].astype(str) == "sliding_action_safe")
    )


# =============================================================================
# Quality-aware balancing (unchanged)
# =============================================================================
def balance_train_rows_quality_weighted(train_rows: pd.DataFrame) -> pd.DataFrame:
    df = train_rows.copy()
    if "q_score" not in df.columns:
        df["q_score"] = 1.0

    w = np.clip(df["q_score"].to_numpy(dtype=np.float32), 0.0, 1.0) ** float(Q_ALPHA)
    w = np.clip(w, 1e-6, None)
    df["_w"] = w

    task_pool = df[(df["task_valid"].astype(int) == 1) & (df["task_target"].astype(int) > 0)].copy()
    rest_pool = df[(df["label_action"].astype(int) == 0) & (df["type"].astype(str) == "sliding_rest_safe")].copy()
    anchor_pool = df[(df["label_action"].astype(int) == 1) & (df["task_valid"].astype(int) == 0)].copy()

    if len(task_pool) == 0:
        out = df.sample(frac=1, random_state=42).reset_index(drop=True)
        out.drop(columns=["_w"], inplace=True, errors="ignore")
        return out

    dist = task_pool["task_target"].value_counts().sort_index()
    target_per_task = int(np.clip(TARGET_TASK_PER_CLASS, 1, TASK_PER_CLASS_CAP))

    balanced_tasks = []
    for t in dist.index:
        td = task_pool[task_pool["task_target"] == t].copy()
        cur = len(td)

        td_good = td[td["q_score"] >= float(Q_MIN_BALANCE)].copy()
        if len(td_good) == 0:
            td_good = td.copy()

        if cur < target_per_task:
            td2 = td_good.sample(n=target_per_task, replace=True, random_state=42, weights=td_good["_w"])
        elif cur > target_per_task:
            td2 = td.sample(n=target_per_task, replace=False, random_state=42, weights=td["_w"])
        else:
            td2 = td

        balanced_tasks.append(td2)

    tasks_bal = pd.concat(balanced_tasks, ignore_index=True)
    n_tasks = len(tasks_bal)

    anchors_keep = anchor_pool
    if len(anchor_pool) > 0 and MAX_ANCHOR_RATIO is not None:
        cap = int(max(0, round(float(MAX_ANCHOR_RATIO) * n_tasks)))
        if cap <= 0:
            anchors_keep = anchor_pool.iloc[:0].copy()
        elif len(anchor_pool) > cap:
            ap_good = anchor_pool[anchor_pool["q_score"] >= float(Q_MIN_BALANCE)].copy()
            if len(ap_good) == 0:
                ap_good = anchor_pool
            anchors_keep = ap_good.sample(n=cap, replace=False, random_state=42, weights=ap_good["_w"])

    n_action = n_tasks + len(anchors_keep)

    r = float(np.clip(TARGET_REST_FRACTION, 0.05, 0.80))
    target_rest = int(round(n_action * r / max(1e-8, (1.0 - r))))
    target_rest = max(int(MIN_REST_SAMPLES), target_rest)

    rest_bal = rest_pool.iloc[:0].copy()
    if len(rest_pool) > 0 and target_rest > 0:
        rest_good = rest_pool[rest_pool["q_score"] >= float(Q_MIN_BALANCE)].copy()
        if len(rest_good) == 0:
            rest_good = rest_pool.copy()

        if len(rest_pool) < target_rest:
            rest_bal = rest_good.sample(n=target_rest, replace=True, random_state=42, weights=rest_good["_w"])
        elif len(rest_pool) > target_rest:
            rest_bal = rest_pool.sample(n=target_rest, replace=False, random_state=42, weights=rest_pool["_w"])
        else:
            rest_bal = rest_pool

    out = pd.concat([tasks_bal, anchors_keep, rest_bal], ignore_index=True)
    out = out.sample(frac=1, random_state=42).reset_index(drop=True)
    out.drop(columns=["_w"], inplace=True, errors="ignore")
    return out


# =============================================================================
# Export core
# =============================================================================
def p5_export_fold(
    splits: pd.DataFrame,
    fold_id: int,
    out_root: Path,
    qc_flag_map: Dict[str, int],
    cache_cap: int = 12,
    mode: str = "balanced",
    causal: bool = False,
) -> None:
    if mode not in ("balanced", "ssl"):
        raise ValueError(f"Unknown mode: {mode}")

    is_balanced = (mode == "balanced")
    out_name = (OUT_NAME_BALANCED if is_balanced else OUT_NAME_SSL) + (LIVE_SUFFIX if causal else "")
    mode_label = ("BALANCED (supervised)" if is_balanced else "SSL (unbalanced)") + (" + LIVE" if causal else "")
    print(f"\n========== FOLD {fold_id} — export mode: {mode_label} ==========")

    fold_rows = splits[splits["fold_id"] == fold_id].copy()
    if fold_rows.empty:
        print(f"[skip] fold {fold_id}: no rows")
        return

    fold_rows = fold_rows[fold_rows["type"].isin(WINDOW_TYPES_EXPORT)].copy()
    if fold_rows.empty:
        print(f"[skip] fold {fold_id}: no rows after WINDOW_TYPES_EXPORT filter")
        return

    fold_rows["use_for_task"] = infer_use_for_task(fold_rows).astype(int)
    fold_rows["task_valid"]   = fold_rows["use_for_task"].astype(int)

    cache = P5CacheLRU(cache_cap)

    print(f"[fold {fold_id}] building per-fold channel schema …")
    schema = p5_build_fold_schema(fold_rows, cache)

    train_stat_rows = fold_rows[fold_rows["split"] == "train"].copy()
    if train_stat_rows.empty:
        print(f"[warn] fold {fold_id}: no TRAIN rows; skipping")
        return

    lim = p5_limit_for_split("train")
    if lim:
        train_stat_rows = train_stat_rows.iloc[:lim]

    stats = {}
    if INCLUDE_EEG:
        stats[EEG_KEY] = p5_init_stats(len(schema["EEG"]))
    if INCLUDE_EMG:
        stats[EMG_KEY] = p5_init_stats(len(schema["EMG"]))
    if INCLUDE_ET:
        stats[ET_KEY]  = p5_init_stats(len(schema["ET"]))

    print(f"[fold {fold_id}] pass1: accumulating TRAIN stats over {len(train_stat_rows):,} windows …")
    for r in train_stat_rows.itertuples(index=False):
        label_csv = str(r.file)
        cache_npz = str(r.cache_npz) if ("cache_npz" in train_stat_rows.columns and pd.notna(getattr(r, "cache_npz"))) else None
        npz = cache.fetch(label_csv, cache_npz=cache_npz)

        s, e = p5_get_window_bounds(npz, r, causal=causal)

        if INCLUDE_EEG and len(schema["EEG"]) > 0:
            Xe, Me, pres = p5_align_window_to_schema(npz, EEG_KEY, [EEG_MASK_KEY], EEG_CH_KEY, schema["EEG"], s, e)
            if Xe is not None and p5_coverage_ok_present_only(Me, pres, MIN_COVERAGE):
                p5_add_stats(stats[EEG_KEY], Xe, Me)

        if INCLUDE_EMG and len(schema["EMG"]) > 0:
            Xm, Mm, pres = p5_align_window_to_schema(npz, EMG_KEY, [EMG_MASK_KEY], EMG_CH_KEY, schema["EMG"], s, e)
            if Xm is not None and p5_coverage_ok_present_only(Mm, pres, MIN_COVERAGE):
                p5_add_stats(stats[EMG_KEY], Xm, Mm)

        if INCLUDE_ET and len(schema["ET"]) > 0:
            Xt, Mt, pres = p5_align_window_to_schema(npz, ET_KEY, ET_MASK_KEYS, ET_CH_KEY, schema["ET"], s, e)
            if Xt is not None and p5_coverage_ok_present_only(Mt, pres, MIN_COVERAGE):
                p5_add_stats(stats[ET_KEY], Xt, Mt)

    means, stds, cnts = {}, {}, {}
    for key in (EEG_KEY, EMG_KEY, ET_KEY):
        if key in stats:
            m, s2, c = p5_finalize_stats(stats[key])
            means[key], stds[key], cnts[key] = m, s2, c

    fold_dir = out_root / f"{out_name}_fold{fold_id}"
    p5_mkdir(fold_dir)

    # stats json unchanged except version name
    fold_meta = {
        "fold_id": int(fold_id),
        "export_mode": mode,
        "causal_live": bool(causal),
        "qc_policy": QC_POLICY,
        "fixed_window_len": int(FIXED_WINDOW_LEN),
        "min_coverage_present_only": float(MIN_COVERAGE),
        "min_modalities_ok": int(MIN_MODALITIES_OK),
        "window_types_export": WINDOW_TYPES_EXPORT,
        "include_onset_anchor": bool(INCLUDE_ONSET_ANCHOR),
        "commit_meta_exported": bool(EXPORT_COMMIT_META),
        "cache_layout": {
            "hashed_dir": str(CACHE_DIR_HASHED),
            "use_hashed_first": bool(USE_HASHED_CACHE_FIRST),
            "sidecar_fallback": True,
            "suffix_candidates": NPZ_SUFFIX_CANDIDATES,
            "preferred_suffix": NPZ_SUFFIX_PREFERRED,
        },
        "include": {"EEG": INCLUDE_EEG, "EMG": INCLUDE_EMG, "ET": INCLUDE_ET},
        "schema_channels": {"EEG": schema["EEG"], "EMG": schema["EMG"], "ET": schema["ET"]},
        "stats": {},
    }
    for key, ch in [(EEG_KEY, schema["EEG"]), (EMG_KEY, schema["EMG"]), (ET_KEY, schema["ET"])]:
        if key in means:
            fold_meta["stats"][key] = {
                "mean": means[key].tolist(),
                "std":  stds[key].tolist(),
                "cnt":  cnts[key].tolist(),
                "num_channels": int(len(ch)),
            }
    (fold_dir / "stats_fold.json").write_text(json.dumps(fold_meta, indent=2))

    # PASS 2: export shards
    for split in ["train", "val", "test"]:
        rows = fold_rows[fold_rows["split"] == split].copy()
        if rows.empty:
            print(f"[fold {fold_id}] no rows for {split}")
            continue

        lim = p5_limit_for_split(split)
        if lim:
            rows = rows.iloc[:lim].copy()

        # ✅ NEW: add/derive commit meta + win_idx BEFORE any balancing
        rows = p5_prepare_commit_meta_rows(rows)

        # TRAIN balancing (balanced mode only)
        if split == "train" and is_balanced:
            q_scores = []
            for rr in rows.itertuples(index=False):
                label_csv = str(rr.file)
                cache_npz = str(rr.cache_npz) if ("cache_npz" in rows.columns and pd.notna(getattr(rr, "cache_npz"))) else None
                npz = cache.fetch(label_csv, cache_npz=cache_npz)
                s, e = p5_get_window_bounds(npz, rr, causal=causal)

                fb = p5_norm_base(label_csv)
                qc_flag = int(qc_flag_map.get(fb, 1))
                qc_scale = 1.0 if qc_flag == 2 else (0.85 if qc_flag == 1 else 0.0)

                r_eeg = r_emg = r_et = 0.0
                kept_modalities = 0

                if INCLUDE_EEG and len(schema["EEG"]) > 0 and EEG_KEY in means:
                    Xe, Me, pres = p5_align_window_to_schema(npz, EEG_KEY, [EEG_MASK_KEY], EEG_CH_KEY, schema["EEG"], s, e)
                    if Xe is not None and p5_coverage_ok_present_only(Me, pres, MIN_COVERAGE):
                        r_eeg = p5_quality_from_mask_present_only(Me, pres)
                        kept_modalities |= 1

                if INCLUDE_EMG and len(schema["EMG"]) > 0 and EMG_KEY in means:
                    Xm, Mm, pres = p5_align_window_to_schema(npz, EMG_KEY, [EMG_MASK_KEY], EMG_CH_KEY, schema["EMG"], s, e)
                    if Xm is not None and p5_coverage_ok_present_only(Mm, pres, MIN_COVERAGE):
                        r_emg = p5_quality_from_mask_present_only(Mm, pres)
                        kept_modalities |= 2

                if INCLUDE_ET and len(schema["ET"]) > 0 and ET_KEY in means:
                    Xt, Mt, pres = p5_align_window_to_schema(npz, ET_KEY, ET_MASK_KEYS, ET_CH_KEY, schema["ET"], s, e)
                    if Xt is not None and p5_coverage_ok_present_only(Mt, pres, MIN_COVERAGE):
                        r_et = p5_quality_from_mask_present_only(Mt, pres)
                        kept_modalities |= 4

                q = p5_qscore_from_modalities(r_eeg, r_emg, r_et, kept_modalities, qc_scale)
                q_scores.append(q)

            rows = rows.copy()
            rows["q_score"] = np.asarray(q_scores, dtype=np.float32)

            print(f"[fold {fold_id}] applying TRAIN balancing (quality-weighted) …")
            rows = balance_train_rows_quality_weighted(rows)
        else:
            if "q_score" not in rows.columns:
                rows["q_score"] = 1.0

        out_split = fold_dir / split
        p5_mkdir(out_split)

        Ce = int(len(schema["EEG"])) if INCLUDE_EEG else 0
        Cm = int(len(schema["EMG"])) if INCLUDE_EMG else 0
        Ct = int(len(schema["ET"]))  if INCLUDE_ET  else 0

        shard_idx = 1
        buf = defaultdict(list)

        kept_task_counts = defaultdict(int)
        total_kept_split = 0

        def flush_shard():
            nonlocal shard_idx, buf
            if not buf["y_action"]:
                return
            shard_path = out_split / f"{split}_shard_{shard_idx:04d}.npz"

            payload = dict(
                X_EEG=np.stack(buf["X_EEG"]) if buf["X_EEG"] else np.zeros((0, FIXED_WINDOW_LEN, Ce), np.float32),
                X_EMG=np.stack(buf["X_EMG"]) if buf["X_EMG"] else np.zeros((0, FIXED_WINDOW_LEN, Cm), np.float32),
                X_ET =np.stack(buf["X_ET" ]) if buf["X_ET" ] else np.zeros((0, FIXED_WINDOW_LEN, Ct), np.float32),
                M_EEG=np.stack(buf["M_EEG"]) if buf["M_EEG"] else np.zeros((0, FIXED_WINDOW_LEN, Ce), np.float32),
                M_EMG=np.stack(buf["M_EMG"]) if buf["M_EMG"] else np.zeros((0, FIXED_WINDOW_LEN, Cm), np.float32),
                M_ET =np.stack(buf["M_ET" ]) if buf["M_ET" ] else np.zeros((0, FIXED_WINDOW_LEN, Ct), np.float32),

                y_action=np.asarray(buf["y_action"], dtype=np.int8),
                y_task=np.asarray(buf["y_task"], dtype=np.int16),
                task_valid=np.asarray(buf["task_valid"], dtype=np.int8),
                use_for_task=np.asarray(buf["use_for_task"], dtype=np.int8),

                subject_id=np.asarray(buf["subject_id"], dtype=np.int16),
                task_code=np.asarray(buf["task_code"], dtype=np.int16),
                trial_id=np.asarray(buf["trial_id"], dtype=np.int16),
                win_type=np.asarray(buf["win_type"], dtype="U32"),
                win_len=np.asarray(buf["win_len"], dtype=np.int32),

                r_eeg=np.asarray(buf["r_eeg"], dtype=np.float32),
                r_emg=np.asarray(buf["r_emg"], dtype=np.float32),
                r_et=np.asarray(buf["r_et"], dtype=np.float32),
                p_eeg=np.asarray(buf["p_eeg"], dtype=np.float32),
                p_emg=np.asarray(buf["p_emg"], dtype=np.float32),
                p_et=np.asarray(buf["p_et"], dtype=np.float32),
                et_valid_frac=np.asarray(buf["et_valid_frac"], dtype=np.float32),

                kept_modalities=np.asarray(buf["kept_modalities"], dtype=np.int8),
                qc_flag=np.asarray(buf["qc_flag"], dtype=np.int8),
                q_score=np.asarray(buf["q_score"], dtype=np.float32),
                sample_weight=np.asarray(buf["sample_weight"], dtype=np.float32),
            )

            # ✅ NEW: commit meta arrays
            if EXPORT_COMMIT_META:
                payload.update(dict(
                    win_idx=np.asarray(buf["win_idx"], dtype=np.int32),
                    is_transition=np.asarray(buf["is_transition"], dtype=np.int8),
                    dist_to_onset_s=np.asarray(buf["dist_to_onset_s"], dtype=np.float32),
                    onset_idx_nearest=np.asarray(buf["onset_idx_nearest"], dtype=np.int32),
                    onset_time_nearest_s=np.asarray(buf["onset_time_nearest_s"], dtype=np.float32),
                    active_prob_mean=np.asarray(buf["active_prob_mean"], dtype=np.float32),
                    w_label_mean=np.asarray(buf["w_label_mean"], dtype=np.float32),
                ))

            np.savez_compressed(shard_path, **payload)
            shard_idx += 1
            buf = defaultdict(list)

        print(f"[fold {fold_id}] exporting {split} ({len(rows):,} windows, mode={mode}, causal={causal}) …")
        for r in rows.itertuples(index=False):
            label_csv = str(r.file)
            cache_npz = str(r.cache_npz) if ("cache_npz" in rows.columns and pd.notna(getattr(r, "cache_npz"))) else None
            npz = cache.fetch(label_csv, cache_npz=cache_npz)

            s, e = p5_get_window_bounds(npz, r, causal=causal)
            win_len = max(0, e - s)

            fb = p5_norm_base(label_csv)
            qc_flag = int(qc_flag_map.get(fb, 1))
            qc_scale = 1.0 if qc_flag == 2 else (0.85 if qc_flag == 1 else 0.0)

            kept_modalities = 0

            # EEG
            r_eeg = p_eeg = 0.0
            Xe = Me = None
            if INCLUDE_EEG and Ce > 0 and EEG_KEY in means:
                Xe, Me, pres_e = p5_align_window_to_schema(npz, EEG_KEY, [EEG_MASK_KEY], EEG_CH_KEY, schema["EEG"], s, e)
                if Xe is not None and p5_coverage_ok_present_only(Me, pres_e, MIN_COVERAGE):
                    r_eeg = p5_quality_from_mask_present_only(Me, pres_e)
                    p_eeg = p5_present_fraction(pres_e)
                    Xe = p5_zscore_masked(Xe, Me, means[EEG_KEY], stds[EEG_KEY])
                    kept_modalities |= 1
                else:
                    Xe, Me = None, None

            # EMG
            r_emg = p_emg = 0.0
            Xm = Mm = None
            if INCLUDE_EMG and Cm > 0 and EMG_KEY in means:
                Xm, Mm, pres_m = p5_align_window_to_schema(npz, EMG_KEY, [EMG_MASK_KEY], EMG_CH_KEY, schema["EMG"], s, e)
                if Xm is not None and p5_coverage_ok_present_only(Mm, pres_m, MIN_COVERAGE):
                    r_emg = p5_quality_from_mask_present_only(Mm, pres_m)
                    p_emg = p5_present_fraction(pres_m)
                    Xm = p5_zscore_masked(Xm, Mm, means[EMG_KEY], stds[EMG_KEY])
                    kept_modalities |= 2
                else:
                    Xm, Mm = None, None

            # ET
            r_et = p_et = 0.0
            Xt = Mt = None
            et_valid_frac = 0.0
            if INCLUDE_ET and Ct > 0 and ET_KEY in means:
                Xt, Mt, pres_t = p5_align_window_to_schema(npz, ET_KEY, ET_MASK_KEYS, ET_CH_KEY, schema["ET"], s, e)
                if Xt is not None and p5_coverage_ok_present_only(Mt, pres_t, MIN_COVERAGE):
                    r_et = p5_quality_from_mask_present_only(Mt, pres_t)
                    p_et = p5_present_fraction(pres_t)
                    Xt = p5_zscore_masked(Xt, Mt, means[ET_KEY], stds[ET_KEY])
                    kept_modalities |= 4
                else:
                    Xt, Mt = None, None

                if ET_VALID_KEY in npz:
                    ev = p5_slice_time(npz[ET_VALID_KEY], s, e)
                    if ev.size:
                        ev = ev.reshape(-1)
                        et_valid_frac = float(np.mean(ev > 0.5))

            if bin(kept_modalities).count("1") < int(MIN_MODALITIES_OK):
                continue

            q_score = p5_qscore_from_modalities(r_eeg, r_emg, r_et, kept_modalities, qc_scale)
            sample_weight = float(np.clip(q_score, 0.0, 1.0) ** float(Q_ALPHA))

            T_len = int(FIXED_WINDOW_LEN)
            Xe = p5_pad_or_trim(Xe, T_len) if Xe is not None else np.zeros((T_len, Ce), np.float32)
            Xm = p5_pad_or_trim(Xm, T_len) if Xm is not None else np.zeros((T_len, Cm), np.float32)
            Xt = p5_pad_or_trim(Xt, T_len) if Xt is not None else np.zeros((T_len, Ct), np.float32)

            Me = p5_pad_or_trim_mask(Me, T_len) if Me is not None else np.zeros((T_len, Ce), np.float32)
            Mm = p5_pad_or_trim_mask(Mm, T_len) if Mm is not None else np.zeros((T_len, Cm), np.float32)
            Mt = p5_pad_or_trim_mask(Mt, T_len) if Mt is not None else np.zeros((T_len, Ct), np.float32)

            y_action = int(r.label_action)
            y_task   = int(r.task_target)
            task_valid = int(r.task_valid)
            use_for_task = int(r.use_for_task)

            # ---- write main tensors ----
            buf["X_EEG"].append(Xe); buf["M_EEG"].append(Me)
            buf["X_EMG"].append(Xm); buf["M_EMG"].append(Mm)
            buf["X_ET"].append(Xt);  buf["M_ET"].append(Mt)

            buf["y_action"].append(y_action)
            buf["y_task"].append(y_task)
            buf["task_valid"].append(task_valid)
            buf["use_for_task"].append(use_for_task)

            buf["subject_id"].append(int(r.subject_id))
            buf["task_code"].append(int(r.task_code))
            buf["trial_id"].append(int(r.trial_id))
            buf["win_type"].append(str(r.type))
            buf["win_len"].append(int(win_len))

            buf["r_eeg"].append(float(r_eeg))
            buf["r_emg"].append(float(r_emg))
            buf["r_et"].append(float(r_et))
            buf["p_eeg"].append(float(p_eeg))
            buf["p_emg"].append(float(p_emg))
            buf["p_et"].append(float(p_et))
            buf["et_valid_frac"].append(float(et_valid_frac))

            buf["kept_modalities"].append(int(kept_modalities))
            buf["qc_flag"].append(int(qc_flag))
            buf["q_score"].append(float(q_score))
            buf["sample_weight"].append(float(sample_weight))

            # ✅ NEW: commit meta per-window
            if EXPORT_COMMIT_META:
                buf["win_idx"].append(int(getattr(r, "win_idx", -1)))
                buf["is_transition"].append(int(getattr(r, "is_transition", 0)))

                # floats may be nan
                buf["dist_to_onset_s"].append(float(getattr(r, "dist_to_onset_s", np.nan)))
                buf["onset_idx_nearest"].append(int(getattr(r, "onset_idx_nearest", -1)))
                buf["onset_time_nearest_s"].append(float(getattr(r, "onset_time_nearest_s", np.nan)))
                buf["active_prob_mean"].append(float(getattr(r, "active_prob_mean", np.nan)))
                buf["w_label_mean"].append(float(getattr(r, "w_label_mean", np.nan)))

            total_kept_split += 1
            kept_task_counts[int(r.task_target)] += 1

            if len(buf["y_action"]) >= int(SHARD_SIZE):
                flush_shard()

        flush_shard()

        # quick meta diagnostics about commit columns
        dist_finite = float(np.isfinite(rows["dist_to_onset_s"].to_numpy(np.float32, copy=False)).mean()) if EXPORT_COMMIT_META else 0.0
        tr_frac = float(rows["is_transition"].to_numpy(np.int8, copy=False).mean()) if EXPORT_COMMIT_META else 0.0

        split_meta = {
            "qc_policy": QC_POLICY,
            "export_mode": mode,
            "causal_live": bool(causal),
            "num_windows_requested": int(len(rows)),
            "num_windows_exported": int(total_kept_split),
            "shard_size": int(SHARD_SIZE),
            "limit": p5_limit_for_split(split),
            "task_distribution_requested": rows["task_target"].value_counts().sort_index().to_dict(),
            "task_distribution_exported": {int(k): int(v) for k, v in sorted(kept_task_counts.items())},
            "fixed_window_len": int(FIXED_WINDOW_LEN),
            "schema_num_channels": {"EEG": Ce, "EMG": Cm, "ET": Ct},
            "commit_meta_exported": bool(EXPORT_COMMIT_META),
            "commit_meta_dist_finite_frac_in_rows": float(dist_finite),
            "commit_meta_transition_frac_in_rows": float(tr_frac),
            "note": (
                "Task head should use task_valid==1 windows only (T1..T5). "
                "REST windows have task_valid=0 even if y_task=0."
            ),
        }
        (out_split / "split_meta.json").write_text(json.dumps(split_meta, indent=2))

    cache.reset()
    print(f"[fold {fold_id}] done → {out_name}_fold{fold_id}")


# =============================================================================
# Main
# =============================================================================
def p5_run_export() -> None:
    if not SPLITS_CSV.exists():
        raise SystemExit(f"[stop] splits not found: {SPLITS_CSV}")

    print("Loading splits …")
    splits = pd.read_csv(SPLITS_CSV, low_memory=False)

    must = {"file","subject_id","split","type","task_target","label_action","start_idx","end_idx","task_code","trial_id","fold_id"}
    missing = must - set(splits.columns)
    if missing:
        raise SystemExit(f"[stop] splits missing columns: {sorted(missing)}")

    allowset_base, qc_flag_map, qc_cache_map = load_qc_maps()

    if "cache_npz" not in splits.columns and len(qc_cache_map) > 0:
        splits["file_base"] = splits["file"].astype(str).apply(p5_norm_base)
        splits["cache_npz"] = splits["file_base"].map(qc_cache_map).astype(object)
        splits.drop(columns=["file_base"], inplace=True, errors="ignore")
        n_has = int(pd.notna(splits["cache_npz"]).sum())
        print(f"[phase5] injected cache_npz from qc_summary: {n_has}/{len(splits)} rows have explicit cache_npz")

    if allowset_base is not None:
        splits["file_base"] = splits["file"].astype(str).apply(p5_norm_base)
        before = len(splits)
        splits = splits[splits["file_base"].isin(allowset_base)].reset_index(drop=True)
        after = len(splits)
        print(f"[QC] windows before: {before} | after: {after} | dropped: {before-after}")
        splits.drop(columns=["file_base"], inplace=True, errors="ignore")
    else:
        print("[QC] No allowlist filtering applied (QC_POLICY off or allowlist missing and no fallback).")

    before = len(splits)
    splits = splits[splits["type"].astype(str).isin(WINDOW_TYPES_EXPORT)].reset_index(drop=True)
    print(f"[phase5] window types filter: {before} -> {len(splits)} (kept {WINDOW_TYPES_EXPORT})")

    folds = sorted(splits["fold_id"].unique().tolist())
    out_root = DATASET_DIR

    print(f"\nExporting {len(folds)} folds …")
    for k in folds:
        k = int(k)
        if EXPORT_BALANCED:
            p5_export_fold(splits, fold_id=k, out_root=out_root, qc_flag_map=qc_flag_map, cache_cap=12, mode="balanced", causal=False)
        if EXPORT_SSL:
            p5_export_fold(splits, fold_id=k, out_root=out_root, qc_flag_map=qc_flag_map, cache_cap=12, mode="ssl", causal=False)

        if EXPORT_LIVE:
            if EXPORT_BALANCED:
                p5_export_fold(splits, fold_id=k, out_root=out_root, qc_flag_map=qc_flag_map, cache_cap=12, mode="balanced", causal=True)
            if EXPORT_SSL:
                p5_export_fold(splits, fold_id=k, out_root=out_root, qc_flag_map=qc_flag_map, cache_cap=12, mode="ssl", causal=True)

    print(f"\n🎉 Phase 5 {P5_VERSION} complete!")
    if EXPORT_BALANCED:
        print(f"   Supervised (balanced): {OUT_NAME_BALANCED}_fold*")
        if EXPORT_LIVE:
            print(f"   Supervised (balanced + live): {OUT_NAME_BALANCED}{LIVE_SUFFIX}_fold*")
    if EXPORT_SSL:
        print(f"   SSL (unbalanced):      {OUT_NAME_SSL}_fold*")
        if EXPORT_LIVE:
            print(f"   SSL (unbalanced + live): {OUT_NAME_SSL}{LIVE_SUFFIX}_fold*")


# In Jupyter: just call
p5_run_export()


Loading splits …
[QC] allowlist not found: /home/tsultan1/IROS/Data/_dataset_icml_v1/qc_allowlist_v4_passwarn.csv → will fallback to qc_summary rule if possible.
[QC] qc_summary_v4.csv not found → qc_flag map empty; no fallback allowset.
[QC] No allowlist filtering applied (QC_POLICY off or allowlist missing and no fallback).
[phase5] window types filter: 1440750 -> 1440750 (kept ['sliding_action_safe', 'sliding_rest_safe', 'onset_anchor'])

Exporting 30 folds …

========== FOLD 1 — export mode: BALANCED (supervised) ==========
[fold 1] building per-fold channel schema …
[fold 1] pass1: accumulating TRAIN stats over 37,245 windows …
[fold 1] applying TRAIN balancing (quality-weighted) …
[fold 1] exporting train (12,757 windows, mode=balanced, causal=False) …
[fold 1] exporting val (8,273 windows, mode=balanced, causal=False) …
[fold 1] exporting test (2,507 windows, mode=balanced, causal=False) …
[fold 1] done → exports_v6_3c_commitmeta_balanced_core_ok_fold1

========== FOLD 1 — expor

In [4]:
from __future__ import annotations
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple
import json
import numpy as np
import pandas as pd

# ==============================
# CONFIG
# ==============================
DATA_ROOT = Path("/home/tsultan1/IROS/Data/_dataset_icml_v1")
EXP_PREFIX = "exports_v6_3c_commitmeta_balanced_core_ok"   # <-- change if needed
SPLITS = ["train", "val", "test"]

# If you want to limit folds for quick testing:
FOLDS_TO_CHECK: Optional[List[int]] = None  # e.g., [1,3,4] ; None = auto-detect all

# If you want to speed up further, set to True (ordering checks become approximate)
SAMPLE_ONLY = False
SAMPLE_PER_SHARD = 2000  # used only if SAMPLE_ONLY=True

# Required keys for commit/trial sanity
REQ_KEYS = [
    "y_action", "subject_id", "task_code", "trial_id",
    "win_idx", "win_type",
]
ONSET_KEYS = ["dist_to_onset_s", "onset_time_nearest_s"]
OPTIONAL_KEYS = ["onset_idx_nearest", "is_transition", "q_score", "qc_flag"]

# Expected onset-anchor window range (your Phase-3 pre/post often ~[-2, +3])
ONSET_ANCHOR_RANGE = (-2.5, 3.5)

# ==============================
# Helpers
# ==============================
def detect_folds(dataset_dir: Path, exp_prefix: str) -> List[int]:
    folds = []
    for p in dataset_dir.glob(f"{exp_prefix}_fold*"):
        if not p.is_dir():
            continue
        suf = p.name.split("_fold")[-1]
        try:
            k = int(suf)
        except Exception:
            continue
        ok = True
        for sp in SPLITS:
            sd = p / sp
            if (not sd.exists()) or (len(list(sd.glob("*_shard_*.npz"))) == 0):
                ok = False
                break
        if ok:
            folds.append(k)
    return sorted(set(folds))

def _as_1d(a: np.ndarray) -> np.ndarray:
    a = np.asarray(a)
    if a.ndim == 0:
        return a.reshape(1)
    return a.reshape(-1)

def _decode_if_bytes(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr)
    if arr.dtype.kind == "S":  # bytes
        return np.array([x.decode("utf-8", errors="ignore") for x in arr], dtype=object)
    return arr

def _to_float(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr)
    if arr.dtype.kind in ("f", "i", "u"):
        return arr.astype(np.float64, copy=False)
    # handle object/strings
    arr = _decode_if_bytes(arr)
    return pd.to_numeric(arr, errors="coerce").to_numpy(dtype=np.float64, copy=False)

def _to_int(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr)
    if arr.dtype.kind in ("i", "u"):
        return arr.astype(np.int64, copy=False)
    arr = _decode_if_bytes(arr)
    return pd.to_numeric(arr, errors="coerce").fillna(-1).astype(np.int64).to_numpy(copy=False)

def _sample_idx(n: int, k: int, rng: np.random.RandomState) -> np.ndarray:
    if n <= k:
        return np.arange(n, dtype=np.int64)
    return rng.choice(n, size=k, replace=False).astype(np.int64)

def _percentiles(x: np.ndarray) -> Dict[str, float]:
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    x = x[np.isfinite(x)]
    if x.size == 0:
        return {"min": np.nan, "p01": np.nan, "p50": np.nan, "p99": np.nan, "max": np.nan}
    return {
        "min": float(np.min(x)),
        "p01": float(np.percentile(x, 1)),
        "p50": float(np.percentile(x, 50)),
        "p99": float(np.percentile(x, 99)),
        "max": float(np.max(x)),
    }

def _load_shard_fields(npz_path: Path, keys: List[str], sample_only: bool, sample_k: int, rng) -> Tuple[Dict[str, np.ndarray], int]:
    with np.load(npz_path, allow_pickle=False) as z:
        # window count
        if "y_action" in z.files:
            n = int(np.asarray(z["y_action"]).shape[0])
        else:
            # fallback
            n = int(np.asarray(z[z.files[0]]).shape[0])

        idx = _sample_idx(n, min(sample_k, n), rng) if sample_only else slice(None)

        out = {}
        for k in keys:
            if k in z.files:
                out[k] = _as_1d(np.asarray(z[k])[idx])
            else:
                out[k] = None
    return out, n

def _ordering_checks_per_shard(df: pd.DataFrame) -> Dict[str, float]:
    """
    Checks within each (subject_id, task_code, trial_id):
      - center_time_s monotonic (mostly non-decreasing) when sorted by win_idx
      - win_idx monotonic (non-decreasing)
      - onset_time_nearest_s roughly constant within trial (std small), if present
    """
    if df.empty:
        return {
            "order_violation_rate": np.nan,
            "winidx_nonmono_rate": np.nan,
            "onset_time_nonconst_rate": np.nan,
            "n_trials": 0,
            "n_pairs_checked": 0,
        }

    key_cols = ["subject_id", "task_code", "trial_id"]
    have_ot = "onset_time_nearest_s" in df.columns

    viol = 0
    pairs = 0
    winidx_bad = 0
    winidx_pairs = 0
    ot_nonconst = 0
    n_trials = 0

    for _, g in df.groupby(key_cols, sort=False):
        n_trials += 1
        # pick ordering key
        if "win_idx" in g.columns and g["win_idx"].notna().any():
            g = g.sort_values("win_idx", kind="mergesort")
        else:
            g = g.sort_values("_ord", kind="mergesort")

        # win_idx monotonic check
        if "win_idx" in g.columns:
            w = pd.to_numeric(g["win_idx"], errors="coerce").to_numpy(np.float64, copy=False)
            m = np.isfinite(w)
            w = w[m]
            if w.size >= 2:
                d = np.diff(w)
                winidx_pairs += int(w.size - 1)
                winidx_bad += int((d < 0).sum())

        # center_time monotonic check
        ct = pd.to_numeric(g["center_time_s"], errors="coerce").to_numpy(np.float64, copy=False)
        m = np.isfinite(ct)
        ct = ct[m]
        if ct.size >= 2:
            d = np.diff(ct)
            pairs += int(ct.size - 1)
            viol += int((d < -1e-6).sum())

        # onset time const check (if present)
        if have_ot:
            ot = pd.to_numeric(g["onset_time_nearest_s"], errors="coerce").to_numpy(np.float64, copy=False)
            ot = ot[np.isfinite(ot)]
            if ot.size >= 3:
                # small threshold: adjust if your times are noisy
                if float(np.std(ot)) > 1e-3:
                    ot_nonconst += 1

    return {
        "order_violation_rate": float(viol / max(1, pairs)) if pairs > 0 else np.nan,
        "winidx_nonmono_rate": float(winidx_bad / max(1, winidx_pairs)) if winidx_pairs > 0 else np.nan,
        "onset_time_nonconst_rate": float(ot_nonconst / max(1, n_trials)) if n_trials > 0 else np.nan,
        "n_trials": int(n_trials),
        "n_pairs_checked": int(pairs),
    }

# ==============================
# Main check
# ==============================
def sanity_check_phase5_exports(
    data_root: Path,
    exp_prefix: str,
    folds: Optional[List[int]] = None,
    splits: List[str] = SPLITS,
    sample_only: bool = False,
    sample_per_shard: int = 2000,
) -> pd.DataFrame:
    if folds is None:
        folds = detect_folds(data_root, exp_prefix)
    folds = sorted(set(int(f) for f in folds))

    rng = np.random.RandomState(0)

    rows = []
    keys_to_load = REQ_KEYS + ONSET_KEYS + OPTIONAL_KEYS

    for fold in folds:
        fold_dir = data_root / f"{exp_prefix}_fold{fold}"
        if not fold_dir.exists():
            print(f"[skip] fold{fold}: missing {fold_dir}")
            continue

        for split in splits:
            sp_dir = fold_dir / split
            shard_paths = sorted(sp_dir.glob("*_shard_*.npz"))
            if not shard_paths:
                print(f"[skip] fold{fold}/{split}: no shards")
                continue

            # accumulators
            n_total = 0
            n_rest = 0
            n_action = 0

            have_keys = {k: True for k in REQ_KEYS}
            have_onset = {k: True for k in ONSET_KEYS}

            dist_all = []
            dist_anchor = []
            center_finite = 0
            dist_finite = 0
            ot_finite = 0
            onset_idx_ge0 = 0
            onset_idx_seen = 0

            win_type_counts = {}

            # ordering checks accumulated across shards
            viol_sum = 0.0
            winidx_bad_sum = 0.0
            ot_nonconst_sum = 0.0
            trials_sum = 0
            pairs_sum = 0
            winidx_pairs_sum = 0

            for sp in shard_paths:
                fields, n_shard = _load_shard_fields(sp, keys_to_load, sample_only, sample_per_shard, rng)
                n_total += int(n_shard if not sample_only else len(fields["y_action"]) if fields["y_action"] is not None else 0)

                # key presence
                for k in REQ_KEYS:
                    have_keys[k] = have_keys[k] and (fields.get(k, None) is not None)
                for k in ONSET_KEYS:
                    have_onset[k] = have_onset[k] and (fields.get(k, None) is not None)

                if fields["y_action"] is None:
                    continue

                ya = _to_int(fields["y_action"])
                n_rest += int((ya == 0).sum())
                n_action += int((ya == 1).sum())

                # win_type distribution
                if fields.get("win_type", None) is not None:
                    wt = _decode_if_bytes(fields["win_type"])
                    for x, c in zip(*np.unique(wt, return_counts=True)):
                        win_type_counts[str(x)] = win_type_counts.get(str(x), 0) + int(c)

                # onset fields
                if (fields.get("dist_to_onset_s", None) is not None) and (fields.get("onset_time_nearest_s", None) is not None):
                    dist = _to_float(fields["dist_to_onset_s"])
                    ot = _to_float(fields["onset_time_nearest_s"])
                    dist_all.append(dist)

                    m_dist = np.isfinite(dist)
                    m_ot = np.isfinite(ot)
                    dist_finite += int(m_dist.sum())
                    ot_finite += int(m_ot.sum())

                    center = ot + dist
                    center_finite += int(np.isfinite(center).sum())

                    # onset_idx sanity (if present)
                    if fields.get("onset_idx_nearest", None) is not None:
                        oi = _to_int(fields["onset_idx_nearest"])
                        onset_idx_seen += int(m_dist.sum())
                        onset_idx_ge0 += int(((oi >= 0) & m_dist).sum())

                    # anchor range check (if win_type exists)
                    if fields.get("win_type", None) is not None:
                        wt = _decode_if_bytes(fields["win_type"])
                        m_anchor = np.array([("onset" in str(x)) for x in wt], dtype=bool) & m_dist
                        if m_anchor.any():
                            dist_anchor.append(dist[m_anchor])

                    # ordering checks (per shard) — requires trial keys + win_idx + center_time
                    need_for_order = all(fields.get(k, None) is not None for k in ["subject_id","task_code","trial_id","win_idx"])
                    if need_for_order:
                        subj = _decode_if_bytes(fields["subject_id"])
                        task = _decode_if_bytes(fields["task_code"])
                        trial = _decode_if_bytes(fields["trial_id"])
                        win_idx = _to_int(fields["win_idx"])

                        df = pd.DataFrame({
                            "subject_id": subj,
                            "task_code": task,
                            "trial_id": trial,
                            "win_idx": win_idx,
                            "dist_to_onset_s": dist,
                            "onset_time_nearest_s": ot,
                            "_ord": np.arange(len(ya), dtype=np.int64),
                        })
                        df["center_time_s"] = df["onset_time_nearest_s"] + df["dist_to_onset_s"]

                        chk = _ordering_checks_per_shard(df)

                        # accumulate approximately weighted (rates are per-shard; we weight by pairs/trials)
                        if np.isfinite(chk["order_violation_rate"]):
                            viol_sum += chk["order_violation_rate"] * max(1, chk["n_pairs_checked"])
                            pairs_sum += chk["n_pairs_checked"]
                        if np.isfinite(chk["winidx_nonmono_rate"]):
                            winidx_bad_sum += chk["winidx_nonmono_rate"] * max(1, chk.get("n_pairs_checked", 0))
                            winidx_pairs_sum += chk.get("n_pairs_checked", 0)
                        if np.isfinite(chk["onset_time_nonconst_rate"]):
                            ot_nonconst_sum += chk["onset_time_nonconst_rate"] * max(1, chk["n_trials"])
                            trials_sum += chk["n_trials"]

            # finalize stats
            dist_all_cat = np.concatenate(dist_all, axis=0) if dist_all else np.asarray([], dtype=np.float64)
            dist_anchor_cat = np.concatenate(dist_anchor, axis=0) if dist_anchor else np.asarray([], dtype=np.float64)

            dist_stats = _percentiles(dist_all_cat)
            anchor_stats = _percentiles(dist_anchor_cat)

            p_has_all_req = float(all(have_keys.values()))
            p_has_onset = float(all(have_onset.values()))

            p_dist_finite = float(dist_finite / max(1, n_total))
            p_ot_finite = float(ot_finite / max(1, n_total))
            p_center_finite = float(center_finite / max(1, n_total))

            p_onset_idx_ge0_given_dist = float(onset_idx_ge0 / max(1, onset_idx_seen)) if onset_idx_seen > 0 else float("nan")

            order_violation_rate = float(viol_sum / max(1, pairs_sum)) if pairs_sum > 0 else float("nan")
            onset_time_nonconst_rate = float(ot_nonconst_sum / max(1, trials_sum)) if trials_sum > 0 else float("nan")

            # onset-anchor in-range rate (sanity)
            lo, hi = ONSET_ANCHOR_RANGE
            in_range = np.isfinite(dist_anchor_cat) & (dist_anchor_cat >= lo) & (dist_anchor_cat <= hi)
            p_anchor_in_range = float(in_range.mean()) if dist_anchor_cat.size else float("nan")

            rows.append({
                "fold": int(fold),
                "split": str(split),
                "n_windows_seen": int(n_total),
                "n_rest": int(n_rest),
                "n_action": int(n_action),

                "has_all_required_keys": int(p_has_all_req),
                "has_onset_fields": int(p_has_onset),

                "p_dist_to_onset_finite": p_dist_finite,
                "p_onset_time_finite": p_ot_finite,
                "p_center_time_finite": p_center_finite,
                "p_onset_idx_ge0_given_dist": p_onset_idx_ge0_given_dist,

                "order_violation_rate_center_time": order_violation_rate,
                "onset_time_nonconst_rate_trials": onset_time_nonconst_rate,

                "p_onset_anchor_dist_in_range": p_anchor_in_range,

                "dist_min": dist_stats["min"],
                "dist_p01": dist_stats["p01"],
                "dist_p50": dist_stats["p50"],
                "dist_p99": dist_stats["p99"],
                "dist_max": dist_stats["max"],

                "anchor_dist_min": anchor_stats["min"],
                "anchor_dist_p01": anchor_stats["p01"],
                "anchor_dist_p50": anchor_stats["p50"],
                "anchor_dist_p99": anchor_stats["p99"],
                "anchor_dist_max": anchor_stats["max"],

                "win_type_counts_json": json.dumps(win_type_counts),
            })

            # quick console summary per split
            print(
                f"[fold{fold} {split}] n={n_total} rest={n_rest} act={n_action} "
                f"| onset_finite(dist/ot/center)={p_dist_finite:.3f}/{p_ot_finite:.3f}/{p_center_finite:.3f} "
                f"| order_viol={order_violation_rate:.6f} "
                f"| anchor_in_range={p_anchor_in_range if np.isfinite(p_anchor_in_range) else float('nan'):.3f}"
            )

    return pd.DataFrame(rows)

# ==============================
# RUN
# ==============================
folds = FOLDS_TO_CHECK if FOLDS_TO_CHECK is not None else None
df = sanity_check_phase5_exports(
    data_root=DATA_ROOT,
    exp_prefix=EXP_PREFIX,
    folds=folds,
    splits=SPLITS,
    sample_only=SAMPLE_ONLY,
    sample_per_shard=SAMPLE_PER_SHARD,
)

out_csv = DATA_ROOT / "phase5_commitmeta_sanity_report.csv"
df.to_csv(out_csv, index=False)
print("\n[done] wrote:", out_csv)

# Show the most important flags quickly:
cols = [
    "fold","split","n_windows_seen","n_rest","n_action",
    "has_all_required_keys","has_onset_fields",
    "p_dist_to_onset_finite","p_onset_time_finite","p_center_time_finite",
    "p_onset_idx_ge0_given_dist",
    "order_violation_rate_center_time",
    "p_onset_anchor_dist_in_range",
    "dist_p01","dist_p50","dist_p99",
]
print(df[cols].sort_values(["split","fold"]).head(30).to_string(index=False))


[fold1 train] n=12757 rest=5103 act=7654 | onset_finite(dist/ot/center)=0.917/0.917/0.917 | order_viol=0.000000 | anchor_in_range=1.000
[fold1 val] n=8273 rest=2953 act=5320 | onset_finite(dist/ot/center)=0.931/0.931/0.931 | order_viol=0.000000 | anchor_in_range=1.000
[fold1 test] n=2507 rest=686 act=1821 | onset_finite(dist/ot/center)=0.947/0.947/0.947 | order_viol=0.000000 | anchor_in_range=1.000
[fold3 train] n=12732 rest=5093 act=7639 | onset_finite(dist/ot/center)=0.920/0.920/0.920 | order_viol=0.000000 | anchor_in_range=1.000
[fold3 val] n=9084 rest=3022 act=6062 | onset_finite(dist/ot/center)=0.937/0.937/0.937 | order_viol=0.000000 | anchor_in_range=1.000
[fold3 test] n=2822 rest=704 act=2118 | onset_finite(dist/ot/center)=0.955/0.955/0.955 | order_viol=0.000000 | anchor_in_range=1.000
[fold4 train] n=12782 rest=5113 act=7669 | onset_finite(dist/ot/center)=0.918/0.918/0.918 | order_viol=0.000000 | anchor_in_range=1.000
[fold4 val] n=7646 rest=2806 act=4840 | onset_finite(dist/ot